# Mini Project — Student Performance Prediction

## Problem Statement
Predict whether a student will **pass** (final exam score >= 60) based on
study habits, attendance, prior performance, and support factors at home.
This is a binary classification problem: can we flag at-risk students
*before* the final exam, using information available earlier in the term?

## Dataset Summary
`../data/student_performance_W5D5.csv` — 350 students, 7 input features
(4 numeric, 3 categorical) plus the raw `final_exam_score` and the
derived binary target `passed`.

| Column | Type | Description |
|---|---|---|
| study_hours_per_week | numeric | Average hours studied per week |
| attendance_percentage | numeric | % of classes attended (has missing values) |
| previous_exam_score | numeric | Score on the prior exam (0-100) |
| sleep_hours | numeric | Average nightly sleep (has missing values) |
| parental_support | categorical | Low / Medium / High (has missing values) |
| internet_access | categorical | Yes / No |
| extracurricular_activities | categorical | Yes / No |
| final_exam_score | numeric | **Not a feature** - this is what `passed` is derived from |
| passed | target | 1 if final_exam_score >= 60, else 0 |

**Important:** `final_exam_score` is dropped before modeling. Keeping it in
`X` would leak the answer directly into the model, since `passed` is
computed from it.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
)

sns.set_theme(style="whitegrid")

## 1. Data Loading

In [3]:
df = pd.read_csv("student_performance_W5D5.csv")
print("Shape:", df.shape)
df.head()

Shape: (350, 9)


,study_hours_per_week,attendance_percentage,previous_exam_score,sleep_hours,parental_support,internet_access,extracurricular_activities,final_exam_score,passed
0,13.8,75.0,49.3,7.0,High,No,No,45.5,0
1,19.6,84.8,59.1,7.1,Medium,Yes,Yes,71.5,1
2,3.1,69.0,68.2,6.4,Medium,Yes,No,74.1,1
3,20.4,87.3,100.0,8.3,High,Yes,No,82.8,1
4,11.8,89.5,53.0,7.3,Low,Yes,Yes,49.8,0


## 2. Exploration

Check class balance, missing values, and how the numeric features relate
to passing before doing any preprocessing.

In [3]:
print("Class balance:\n", df["passed"].value_counts(normalize=True).round(3))
print("\nMissing values:\n", df.isna().sum())
df.describe().T[["mean", "std", "min", "max"]]

Class balance:
 passed
1    0.554
0    0.446
Name: proportion, dtype: float64

Missing values:
 study_hours_per_week           0
attendance_percentage         12
previous_exam_score            0
sleep_hours                    9
parental_support               7
internet_access                0
extracurricular_activities     0
final_exam_score               0
passed                         0
dtype: int64


,mean,std,min,max
study_hours_per_week,11.746857,4.463623,0.0,23.6
attendance_percentage,80.853846,11.683207,50.3,100.0
previous_exam_score,66.297714,14.402031,30.4,100.0
sleep_hours,6.915836,1.200410,3.1,9.9
final_exam_score,61.970286,12.385900,28.7,94.4
passed,0.554286,0.497756,0.0,1.0


In [4]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(data=df, x="passed", y="previous_exam_score", ax=axes[0])
axes[0].set_title("Previous Exam Score by Outcome")
sns.boxplot(data=df, x="passed", y="attendance_percentage", ax=axes[1])
axes[1].set_title("Attendance % by Outcome")
plt.tight_layout()
plt.savefig("../plots/feature_boxplots_W5D5.png", dpi=150)
plt.show()

## 3. Preprocessing

Three things to handle:
1. Drop `final_exam_score` (leakage - see note above) and separate
   features (`X`) from the target (`y`).
2. Impute missing values - median for numeric columns, most-frequent for
   the categorical column.
3. Scale numeric features and one-hot encode categorical features, bundled
   into a single `ColumnTransformer` so both happen consistently.

In [5]:
X = df.drop(columns=["passed", "final_exam_score"])
y = df["passed"]

numeric_features = ["study_hours_per_week", "attendance_percentage", "previous_exam_score", "sleep_hours"]
categorical_features = ["parental_support", "internet_access", "extracurricular_activities"]

preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_features),
])
print("Feature columns:", list(X.columns))

Feature columns: ['study_hours_per_week', 'attendance_percentage', 'previous_exam_score', 'sleep_hours', 'parental_support', 'internet_access', 'extracurricular_activities']


## 4. Train-Test Split

20% held out as a test set, stratified on `passed` so both classes keep
their proportions in train and test.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")

Train size: 280 | Test size: 70


## 5. Training

Two models, each wrapped with the same preprocessing pipeline so imputing,
scaling, and encoding happen identically for both:
- **Logistic Regression** — a linear baseline.
- **Random Forest** — an ensemble of trees, usually more flexible with
  non-linear relationships and feature interactions.

In [7]:
models = {
    "Logistic Regression": Pipeline([
        ("preprocessing", preprocessor),
        ("classifier", LogisticRegression(max_iter=5000)),
    ]),
    "Random Forest": Pipeline([
        ("preprocessing", preprocessor),
        ("classifier", RandomForestClassifier(n_estimators=100, random_state=42)),
    ]),
}

for model in models.values():
    model.fit(X_train, y_train)

print("Both models trained.")

Both models trained.


## 6. Prediction

In [8]:
predictions = {name: model.predict(X_test) for name, model in models.items()}

comparison_df = pd.DataFrame({
    "actual": y_test.values[:10],
    **{name: preds[:10] for name, preds in predictions.items()},
})
comparison_df

,actual,Logistic Regression,Random Forest
0,1,1,1
1,1,0,1
2,1,1,1
3,1,0,0
4,0,1,1
5,0,0,1
6,0,1,1
7,1,0,0
8,1,1,1
9,0,0,0


## 7. Evaluation

Accuracy alone can be misleading on a dataset this close to balanced
(55%/45%), so precision, recall, and F1 are tracked alongside it, plus a
5-fold cross-validated accuracy as a sanity check beyond the single split.

In [9]:
results = {}
for name, model in models.items():
    preds = predictions[name]
    cv_scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
    results[name] = {
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds),
        "recall": recall_score(y_test, preds),
        "f1": f1_score(y_test, preds),
        "cv_mean_accuracy": cv_scores.mean(),
    }

results_df = pd.DataFrame(results).T.round(4)
results_df

,accuracy,precision,recall,f1,cv_mean_accuracy
Logistic Regression,0.7429,0.7692,0.7692,0.7692,0.7714
Random Forest,0.7143,0.7317,0.7692,0.7500,0.7229


In [10]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, preds) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Fail", "Pass"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(name)
fig.suptitle("Confusion Matrices")
fig.tight_layout()
fig.savefig("../plots/confusion_matrices_W5D5.png", dpi=150)
plt.show()

In [11]:
for name, preds in predictions.items():
    print(f"=== {name} ===")
    print(classification_report(y_test, preds, target_names=["Fail", "Pass"]))

=== Logistic Regression ===
              precision    recall  f1-score   support

        Fail       0.71      0.71      0.71        31
        Pass       0.77      0.77      0.77        39

    accuracy                           0.74        70
   macro avg       0.74      0.74      0.74        70
weighted avg       0.74      0.74      0.74        70

=== Random Forest ===
              precision    recall  f1-score   support

        Fail       0.69      0.65      0.67        31
        Pass       0.73      0.77      0.75        39

    accuracy                           0.71        70
   macro avg       0.71      0.71      0.71        70
weighted avg       0.71      0.71      0.71        70



## 8. Feature Importance (Random Forest)

Which features the Random Forest actually relies on, to sanity-check that
it learned something sensible rather than noise.

In [12]:
rf_model = models["Random Forest"].named_steps["classifier"]
feature_names = (
    numeric_features
    + list(models["Random Forest"].named_steps["preprocessing"]
           .named_transformers_["cat"].named_steps["onehot"]
           .get_feature_names_out(categorical_features))
)
importances = pd.Series(rf_model.feature_importances_, index=feature_names)
top_features = importances.sort_values(ascending=False).head(8)

plt.figure(figsize=(7, 5))
top_features.sort_values().plot(kind="barh", color="#55a868")
plt.xlabel("Importance")
plt.title("Top Features Driving the Random Forest")
plt.tight_layout()
plt.savefig("../plots/feature_importance_W5D5.png", dpi=150)
plt.show()

print(top_features)

previous_exam_score        0.257377
study_hours_per_week       0.240063
attendance_percentage      0.181362
sleep_hours                0.144095
parental_support_Low       0.042488
parental_support_High      0.041966
internet_access_No         0.022459
parental_support_Medium    0.019182
dtype: float64


## 9. Model Comparison and Conclusion

In [13]:
best_model = results_df["f1"].idxmax()
print(f"Best model by F1-score: {best_model}")
print(results_df)

Best model by F1-score: Logistic Regression
                     accuracy  precision  recall      f1  cv_mean_accuracy
Logistic Regression    0.7429     0.7692  0.7692  0.7692            0.7714
Random Forest          0.7143     0.7317  0.7692  0.7500            0.7229


**Conclusion:** both models perform reasonably well on this synthetic,
moderately balanced dataset. `previous_exam_score` and `attendance_percentage`
dominate feature importance for the Random Forest, which matches how the
data was constructed - prior performance and attendance are the strongest
real-world predictors of passing, too. The small gap (if any) between the
two models matters less than the workflow: define the problem clearly,
watch for leakage (dropping `final_exam_score`), preprocess inside a
pipeline so train/test treatment stays consistent, evaluate with more than
one metric, and cross-validate rather than trusting a single split.